# Статистическая проверка гипотез: Математика здравого смысла

Визуализация подсказывает *"где искать"*, а статистика отвечает на вопрос *"правда ли это"*.

В этом ноутбуке мы на реальных данных сети «Пятёрочка» научимся доказывать закономерности, используя научный подход.

### Ссылка на данные
Мы будем работать с реальным файлом .xlsx: `https://dano.hse.ru/mirror/pubs/share/1102761259.xlsx`

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats # Библиотека для научных вычислений

%matplotlib inline
sns.set_theme(style="whitegrid")

# Загружаем данные напрямую по ссылке
url = 'https://dano.hse.ru/mirror/pubs/share/1102761259.xlsx'
df = pd.read_excel(url)

# Удалим пропуски, чтобы статистика не ломалась
df_clean = df.dropna(subset=['Трафик (модифицирован)', 'Средний чек (модифицирован)', 'Регион']).copy()

df_clean.head()

KeyError: ['Трафик (модифицирован)', 'Средний чек (модифицирован)']

# Часть 1. Механика проверки гипотез

Любой статистический тест проходит по одному сценарию. Представьте, что мы в суде.

1.  **Формулируем Нулевую гипотезу ($H_0$):** "Подсудимый невиновен". В мире данных это значит: **"Различий нет, эффекта нет, это все случайность"**.
2.  **Формулируем Альтернативную гипотезу ($H_1$):** "Подсудимый виновен". В данных: **"Различия реальны и закономерны"**.
3.  **Считаем p-value (Probability Value):** Это вероятность увидеть наши данные (или еще более экстремальные), если $H_0$ верна.

### Как читать p-value?
*   Если **p-value < 0.05** (менее 5%): Шанс, что это случайность, ничтожно мал. Мы **отвергаем $H_0$**. (Разница доказана).
*   Если **p-value > 0.05**: Мы не можем отвергнуть $H_0$. (Может разница есть, но доказательств мало).

---

# Часть 2. Проверка на нормальность (Тест Шапиро-Уилка)

Многие мощные тесты (например, Стьюдента) требуют, чтобы данные были распределены **нормально** (похожи на "колокол" Гаусса).

**Суть теста:** Сравнивает, насколько форма гистограммы ваших данных отклоняется от идеального математического колокола.

*   $H_0$: Данные распределены нормально.
*   $H_1$: Данные НЕ распределены нормально.

*Задача: Проверим распределение Трафика.*

In [ ]:
# Строим гистограмму + QQ-plot (график квантилей) для визуальной оценки
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df_clean['Трафик (модифицирован)'], kde=True, ax=ax[0], color='teal')
ax[0].set_title('Гистограмма')

stats.probplot(df_clean['Трафик (модифицирован)'], dist="norm", plot=ax[1])
ax[1].set_title('Q-Q Plot (точки должны лежать на красной линии)')

plt.show()

# Тест Шапиро-Уилка
# Берем случайные 1000 значений, так как тест чувствителен к очень большим выборкам
stat, p_value = stats.shapiro(df_clean['Трафик (модифицирован)'].sample(1000, random_state=42))

print(f"Статистика: {stat:.3f}, p-value: {p_value:.10f}")

if p_value < 0.05:
    print("ВЫВОД: Распределение НЕ нормальное (отвергаем H0).")
else:
    print("ВЫВОД: Распределение похоже на нормальное.")

> **Важно:** Реальные бизнес-данные (деньги, трафик) почти никогда не бывают идеально нормальными. Часто мы видим "длинные хвосты" справа.

---

# Часть 3. Сравнение двух групп (T-тест vs Манн-Уитни)

Предположим, мы хотим сравнить **Трафик** в двух регионах: `Москва` и `Санкт-Петербург` (или любой другой регион).

### Вариант А. T-test Стьюдента
**Суть теста:** Сравнивает **средние значения** двух выборок, учитывая их разброс (дисперсию).
Формула грубо: $T = \frac{Разница\_Средних}{Шум\_(дисперсия)}$.

**Требование:** Значения внутри каждой группы должны быть распределены **нормально** (особенно важно для маленьких выборок N < 30). Если выборок очень много, T-тест устойчив к ненормальности (ЦПТ), но лучше не рисковать.

### Вариант Б. U-критерий Манна-Уитни
**Суть теста:** Не смотрит на сами числа! Он выстраивает всех людей (магазины) в одну шеренгу по росту (ранжирует) и смотрит: правда ли, что магазины Москвы чаще стоят в начале шеренги, а Питера — в конце?
**Требование:** Ему плевать на распределение и выбросы.

*Задача: Сравним трафик в Ростове-на-Дону и Чехове (пример).* 

In [ ]:
# Выбираем два региона для сравнения (поменяйте на свои, если хотите)
region_1 = "Ростов-на-Дону г"
region_2 = "Чехов г"

group_1 = df_clean[df_clean['Населённый пункт'] == region_1]['Трафик (модифицирован)']
group_2 = df_clean[df_clean['Населённый пункт'] == region_2]['Трафик (модифицирован)']

print(f"Магазинов в {region_1}: {len(group_1)}")
print(f"Магазинов в {region_2}: {len(group_2)}")

# Визуализация
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean[df_clean['Населённый пункт'].isin([region_1, region_2])], 
            x='Населённый пункт', y='Трафик (модифицирован)')
plt.show()

# Применяем Манна-Уитни (так как мы выяснили выше, что трафик распределен ненормально)
u_stat, p_val = stats.mannwhitneyu(group_1, group_2)

print(f"P-value: {p_val:.5f}")

if p_val < 0.05:
    print("ВЫВОД: Разница статистически значима! Один город явно 'качает' лучше.")
else:
    print("ВЫВОД: Разница, скорее всего, случайна. Данные не позволяют сказать, кто лучше.")

---

# Часть 4. Сравнение 3+ групп (ANOVA)

Что если мы хотим сравнить трафик сразу во всех регионах (или типах торговых площадей)? Попарно сравнивать нельзя (накапливается ошибка).

### ANOVA (Дисперсионный анализ)
**Суть теста:** Сравнивает отношение Сигнала к Шуму.
*   **Сигнал:** Насколько сильно отличаются средние чеки *между* группами (маленькие магазины vs большие).
*   **Шум:** Насколько сильно чеки скачут *внутри* каждой группы.

Если (Разброс МЕЖДУ) >> (Разброс ВНУТРИ), значит группы реально разные.

*   $H_0$: Средние всех групп равны.
*   $H_1$: Хотя бы одно среднее отличается.

*Задача: Влияет ли категория `Торговая площадь` (Категории 'маленький', 'средний', 'большой') на `Средний чек`?*

In [ ]:
# Посмотрим, какие есть категории площади
print(df_clean['Торговая площадь, категориальный'].unique())

# Подготовим данные: это список списков (каждый список - чеки для конкретной категории площади)
groups = []
for cat in df_clean['Торговая площадь, категориальный'].unique():
    groups.append(df_clean[df_clean['Торговая площадь, категориальный'] == cat]['Средний чек (модифицирован)'])

# Краскел-Уоллис (Kruskal-Wallis)
# Это аналог ANOVA для ненормальных распределений (как Манн-Уитни, только для 3+ групп)
# Мы используем его, чтобы быть честными с данными
stat, p_val = stats.kruskal(*groups)

print(f"Kruskal-Wallis p-value: {p_val}")

if p_val < 0.05:
    print("ВЫВОД: Размер магазина влияет на средний чек (статистически значимо).")
else:
    print("ВЫВОД: Нет доказательств, что размер магазина влияет на чек.")

# Визуализируем результат
plt.figure(figsize=(10, 6))
sns.boxplot(x='Торговая площадь, категориальный', y='Средний чек (модифицирован)', data=df_clean, palette='Set3')
plt.show()

---

# Часть 5. Корреляция (Пирсон vs Спирман)

**Суть:** Мы хотим понять, как `X` влияет на `Y`.

1.  **Пирсон (Pearson):** Проверяет **линейную** связь ($y = kx + b$). Требует нормальности данных.
2.  **Спирман (Spearman):** Проверяет **монотонную** связь (если растет X, то растет и Y, но, возможно, не по прямой линии, а по дуге). Основан на рангах. Более универсален.

*   $H_0$: Связи нет (коэффициент корреляции = 0).
*   $H_1$: Связь есть.

*Задача: Влияет ли `Численность населения` в городе на `Трафик` магазина?*

In [ ]:
# Корреляция Спирмана (так как население может меняться экспоненциально)
corr, p_val = stats.spearmanr(df_clean['Численность населения'], df_clean['Трафик (модифицирован)'])

print(f"Коэффициент корреляции: {corr:.3f}")
print(f"P-value: {p_val}")

if p_val < 0.05:
    if abs(corr) > 0.3:
        print("ВЫВОД: Связь есть, и она заметная.")
    else:
        print("ВЫВОД: Связь статистически есть, но она очень слабая (еле заметная).")
else:
    print("ВЫВОД: Связи нет.")
    
    
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_clean, x='Численность населения', y='Трафик (модифицирован)', alpha=0.3)
plt.xscale('log') # Логарифмическая шкала, так как население сильно разнится
plt.title('Трафик vs Население (Log scale)')
plt.show()

# Резюме

| Вопрос | Данные "Нормальные" | Данные "Кривые" / Выбросы |
| --- | --- | --- |
| **Разница в 2 группах** | T-test Стьюдента | Тест Манна-Уитни |
| **Разница в 3+ группах** | ANOVA | Kruskal-Wallis |
| **Взаимосвязь (Корреляция)** | Пирсон (Line) | Спирман (Rank) |